Catboost:  

model w best param with 14 feature dataset:
{'learning_rate': 0.055710965281825744,
                      'depth': 4,
                      'l2_leaf_reg': 16.378726060483704,
                      'random_strength': 5.418648662113677,
                      'subsample': 0.930094231107212,
                      'rsm': 0.689140498436157,
                      'border_count': 254,
                      'one_hot_max_size': 4}

model w best param with 14 feature dataset:

In [1]:
import numpy as np, pandas as pd, optuna
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score

In [2]:
TRAIN_FE_PATH = "train_fe.csv"          # dataset train dengan 17 fitur
SEED = 42
N_TRIALS = 30
DB_PATH = "sqlite:///catboost_optuna.db"   # menyimpan progres; bisa dilanjutkan jika terhenti

CAT_COLS = ["Gender", "City_Type", "Current_Car_Type",
            "Home_Charging_Possible", "Subsidy_Available", "Range_Anxiety_Level"]

# Titik pusat pencarian (hasil tuning sebelumnya)
CENTER = {'learning_rate': 0.055710965281825744, 'depth': 4, 'l2_leaf_reg': 16.378726060483704,
          'random_strength': 5.418648662113677, 'subsample': 0.930094231107212,
          'rsm': 0.689140498436157, 'border_count': 254, 'one_hot_max_size': 4}

# Hasil 5-fold CV sebelumnya (fitur sama, parameter CENTER) untuk pembanding
REF_FOLD_AUC = [0.94088, 0.94179, 0.94309, 0.94271, 0.94207]

In [3]:
# ---------- 1. Data ----------
df = pd.read_csv(TRAIN_FE_PATH)
y = (df.pop("Will_Buy_EV") == "Yes").astype(int)
X = df.drop(columns="id")
print("Jumlah fitur:", X.shape[1], "| baris:", X.shape[0])

# satu holdout stratified 80/20 untuk tuning
X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
train_pool = Pool(X_tr, y_tr, cat_features=CAT_COLS)     # dibuat sekali, dipakai semua trial
valid_pool = Pool(X_va, y_va, cat_features=CAT_COLS)

Jumlah fitur: 17 | baris: 668665


In [4]:
# ---------- 2. Optuna: pencarian di sekitar CENTER ----------
def objective(trial):
    params = dict(
        loss_function="Logloss", eval_metric="AUC",
        iterations=5000, early_stopping_rounds=100,
        bootstrap_type="Bernoulli",
        learning_rate=trial.suggest_float("learning_rate", 0.035, 0.085, log=True),
        depth=trial.suggest_int("depth", 3, 6),                       # sebelumnya 4 = batas bawah ruang lama
        l2_leaf_reg=trial.suggest_float("l2_leaf_reg", 8, 30, log=True),
        random_strength=trial.suggest_float("random_strength", 2, 10, log=True),
        subsample=trial.suggest_float("subsample", 0.8, 1.0),
        rsm=trial.suggest_float("rsm", 0.5, 0.85),
        border_count=trial.suggest_categorical("border_count", [128, 254]),
        one_hot_max_size=trial.suggest_categorical("one_hot_max_size", [2, 3, 4]),   # kolom kategorik maks 4 nilai
        random_seed=SEED, verbose=0, thread_count=-1,
    )
    m = CatBoostClassifier(**params)
    m.fit(train_pool, eval_set=valid_pool, use_best_model=True)
    trial.set_user_attr("best_iter", int(m.get_best_iteration()))
    return m.get_best_score()["validation"]["AUC"]


study = optuna.create_study(
    direction="maximize", study_name="catboost_17feat",
    sampler=optuna.samplers.TPESampler(seed=SEED),
    storage=DB_PATH, load_if_exists=True,
)
if len(study.trials) == 0:
    study.enqueue_trial(CENTER)          # trial pertama = parameter lama, sebagai acuan di data 17 fitur

remaining = N_TRIALS - len(study.trials)
if remaining > 0:
    study.optimize(objective, n_trials=remaining)

print("\nTrial 0 (parameter lama) holdout AUC:", round(study.trials[0].value, 5))
print("Best holdout AUC :", study.best_value)
print("Best params      :", study.best_params)
print("Best iteration   :", study.best_trial.user_attrs["best_iter"])

[I 2026-09-21 13:18:38,359] A new study created in RDB with name: catboost_17feat
[I 2026-09-21 13:24:25,352] Trial 0 finished with value: 0.9418434948997287 and parameters: {'learning_rate': 0.055710965281825744, 'depth': 4, 'l2_leaf_reg': 16.378726060483704, 'random_strength': 5.418648662113677, 'subsample': 0.930094231107212, 'rsm': 0.689140498436157, 'border_count': 254, 'one_hot_max_size': 4}. Best is trial 0 with value: 0.9418434948997287.
[I 2026-09-21 13:37:42,376] Trial 1 finished with value: 0.9416390693327944 and parameters: {'learning_rate': 0.048797481630675855, 'depth': 6, 'l2_leaf_reg': 21.051226176579352, 'random_strength': 5.241726043075503, 'subsample': 0.8312037280884873, 'rsm': 0.5545980821176709, 'border_count': 254, 'one_hot_max_size': 3}. Best is trial 0 with value: 0.9418434948997287.
[I 2026-09-21 13:45:47,593] Trial 2 finished with value: 0.9411590667345282 and parameters: {'learning_rate': 0.08276060574734101, 'depth': 6, 'l2_leaf_reg': 10.59203179808806, 'ra


Trial 0 (parameter lama) holdout AUC: 0.94184
Best holdout AUC : 0.9419797965838493
Best params      : {'learning_rate': 0.06485099857373895, 'depth': 3, 'l2_leaf_reg': 29.600704854843105, 'random_strength': 2.0170912438050994, 'subsample': 0.9884167852402601, 'rsm': 0.8421685949691121, 'border_count': 254, 'one_hot_max_size': 4}
Best iteration   : 3423


In [ ]:
# ---------- 3. Cek performa: StratifiedKFold 5x dengan best params ----------
final_params = dict(
    loss_function="Logloss", eval_metric="AUC",
    iterations=5000, early_stopping_rounds=200,
    bootstrap_type="Bernoulli",
    random_seed=SEED, verbose=0, thread_count=-1,
    **study.best_params,
)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
oof = np.zeros(len(X))
fold_auc, best_iters = [], []

for k, (tr, va) in enumerate(skf.split(X, y), 1):
    tr_pool = Pool(X.iloc[tr], y.iloc[tr], cat_features=CAT_COLS)
    va_pool = Pool(X.iloc[va], y.iloc[va], cat_features=CAT_COLS)
    m = CatBoostClassifier(**final_params)
    m.fit(tr_pool, eval_set=va_pool, use_best_model=True)
    oof[va] = m.predict_proba(va_pool)[:, 1]
    fold_auc.append(roc_auc_score(y.iloc[va], oof[va]))
    best_iters.append(m.get_best_iteration())
    print(f"fold {k}: AUC={fold_auc[-1]:.5f}  best_iter={best_iters[-1]}", flush=True)

print(f"\nMean AUC = {np.mean(fold_auc):.5f} ± {np.std(fold_auc):.5f}")
print(f"OOF AUC  = {roc_auc_score(y, oof):.5f}")

# pembanding dengan parameter lama (fold identik)
delta = np.array(fold_auc) - np.array(REF_FOLD_AUC)
print(f"Vs parameter lama (mean {np.mean(REF_FOLD_AUC):.5f}): delta per fold = {np.round(delta, 5).tolist()} "
      f"| mean delta = {delta.mean():+.5f} | lebih baik di {(delta > 0).sum()}/5 fold")
print(f"Saran iterations untuk latih di 100% data: ~{int(np.mean(best_iters) * 1.1)}")

fold 1: AUC=0.94089  best_iter=3303
fold 2: AUC=0.94193  best_iter=4002
fold 3: AUC=0.94315  best_iter=3685
fold 4: AUC=0.94280  best_iter=3973
fold 5: AUC=0.94217  best_iter=3869

Mean AUC = 0.94219 ± 0.00078
OOF AUC  = 0.94218
Vs parameter lama (mean 0.94211): delta per fold = [1e-05, 0.00014, 6e-05, 9e-05, 0.0001] | mean delta = +0.00008 | lebih baik di 5/5 fold
Saran iterations untuk latih di 100% data: ~4143


: 